<div class="alert alert-block alert-info">
Before you start running this notebook, create a cloud app in Verily Workbench with a <b>low compute</b> profile. To do so,
<br/>
    
<ul>
    <li>Open the <b>Apps</b> tab in your workspace and click <b>+ New app instance</b>.</li>
    <li>Select <b>R Analysis</b>.</li>
    <li>On the <b>Compute options</b> step, the default profile (2 CPUs, 13 GB of memory) is sufficient.</li>
    <li>For more information, see the <a href="https://support.workbench.verily.com/docs/guides/cloud_apps/gcp/compute_config_options/">Verily Workbench compute profile guide</a>.</li>
</ul>
    
</div>

<h1>Notebook Objectives</h1>

This notebook compares per-sample structural variant counts from PAV, PBSV, and Sniffles on T2T. Stats files are aggregated by SV type and length bin (50–100 bp through 10 kb+) and plotted as boxplots and grouped bars.

<h2>About notebook public versions</h2>

This notebook is a public counterpart of an analysis notebook in the *All of Us* Researcher Workbench (Controlled Tier). Individual-level genomic, EHR, and derived files are not included in this repository. In these public notebook versions, output cells have also been removed.

Where a cell reads or writes data files, the path is written relative to `AOU_LR_DATA`, an environment variable that this notebook reads but never sets. Before running, set it to the location of the notebook's data (in the original workspace, the workspace bucket). For run details, see the version of this notebook in the Researcher Workbench Controlled Tier workspace.

In [ ]:
install.packages("janitor")

In [ ]:
library(reticulate)
library(bigrquery)
library(ggplot2)
library(tidyverse)
library(ggpubr)
library(vcfR)
library(janitor)

In [ ]:
# The FISS (FireCloud) Python API, called from R through reticulate, for reading Terra data tables.
fapi = import('firecloud.api')

In [ ]:
# Workspace details Terra sets as environment variables.
BILLING_PROJECT_ID = Sys.getenv('GOOGLE_PROJECT')       # the Google billing project
WORKSPACE_BUCKET = Sys.getenv('WORKSPACE_BUCKET')       # the workspace bucket
WORKSPACE_NAME = Sys.getenv('WORKSPACE_NAME')           # workspace name
WORKSPACE_NAMESPACE = Sys.getenv('WORKSPACE_NAMESPACE') # workspace namespace

In [ ]:
# Load a Terra data table into a tibble, converting 'true'/'false' list columns to logicals.
load_workspace_table <- function(table_name) {
    a = fapi$get_entities(WORKSPACE_NAMESPACE, WORKSPACE_NAME, table_name)

    json = a$json()
    df = tibble(metadata = json) %>% unnest_wider(metadata) %>% unnest_wider(attributes)

    for (name in df %>% select(where(is.list)) %>% names()) {
        q = as.vector(unlist(df[,name])) %in% c("true", "TRUE", "false", "FALSE") %>% unique()
        if (q) {
            df[,name] = as.logical(as.vector(unlist(df[,name])))
        }
    }
    
    return(df)
}

In [ ]:
# Per-sample table for the T2T-aligned samples; the *_structural_variant_stats columns point to each
# caller's SV summary (counts by SV type and length bin).
ha_sample_t2t_df = load_workspace_table('ha-sample-t2t')

In [ ]:
ha_sample_t2t_df

In [ ]:
# PAV SV stats for one sample, with a Total row added over the length bins.
system(paste("gsutil cp ", ha_sample_t2t_df$pav_structural_variant_stats[1], " .", sep=""), intern=TRUE)

pav_df = read_table(basename(ha_sample_t2t_df$pav_structural_variant_stats[1]))
pav_df

In [ ]:
pav_df %>% tail(nrow(pav_df) - 1) %>%
           adorn_totals("row") %>%
           mutate(sample = gsub("pav_", "", gsub(".stats.txt", "", basename(ha_sample_t2t_df$pav_structural_variant_stats[1]))))


In [ ]:
# Download every sample's PAV SV stats.
for (file in (ha_sample_t2t_df %>% filter(!is.na(pav_structural_variant_stats)) %>% pull(pav_structural_variant_stats))) {
    #if (!file.exists(basename(file))) {
        print(file)
        system(paste("gsutil cp ", file, " .", sep=""), intern=TRUE)
    #}
}

In [ ]:
system("ls -lah | grep pav", intern=TRUE)

In [ ]:
# Combine all samples' PAV stats: the four length-bin rows plus a Total row per sample.
pav_all_sv_df = NULL
for (file in (ha_sample_t2t_df %>% filter(!is.na(ha_sample_t2t_df$pav_structural_variant_stats)) %>% pull(pav_structural_variant_stats))) {
    pav_file = basename(file)
    if (file.exists(pav_file)) {
        pav_df = read_table(pav_file, show_col_types=FALSE)
        pav_df = pav_df %>% tail(n=4) %>% adorn_totals("row") %>% mutate(sample = pav_file)
        
        if (is.null(pav_all_sv_df)) {
            pav_all_sv_df = pav_df
        } else {
            pav_all_sv_df = rbind(pav_all_sv_df, pav_df)
        }
    }
}

In [ ]:
pav_df

In [ ]:
# One sample's PAV SV counts by type, for each length bin and in total.
par(mar=c(5, 7, 4, 2))
pav_df %>% filter(Len == '50-100bp') %>% select(Del, Dup, Inv, INS, TRA, UNK) %>% as.matrix() %>% barplot(las=1, cex.axis=1.3, main='50-100 bp', ylim=c(0, 10000))
pav_df %>% filter(Len == '100-1000bp') %>% select(Del, Dup, Inv, INS, TRA, UNK) %>% as.matrix() %>% barplot(las=1, cex.axis=1.3, main='100-1000 bp', ylim=c(0, 10000))
pav_df %>% filter(Len == '1000-10000bp') %>% select(Del, Dup, Inv, INS, TRA, UNK) %>% as.matrix() %>% barplot(las=1, cex.axis=1.3, main='1000-10000 bp', ylim=c(0, 10000))
pav_df %>% filter(Len == '10000+bp') %>% select(Del, Dup, Inv, INS, TRA, UNK) %>% as.matrix() %>% barplot(las=1, cex.axis=1.3, main='10000+bp', ylim=c(0, 10000))
pav_df %>% filter(Len == 'Total') %>% select(Del, Dup, Inv, INS, TRA, UNK) %>% as.matrix() %>% barplot(las=1, cex.axis=1.3, main='Total', ylim=c(0, 10000))


In [ ]:
pav_df

In [ ]:
pav_df$Len

In [ ]:
# Distribution across samples of PAV SV counts by type, one box plot per length bin.
for (len in pav_df$Len) {
    l = list(
        'Del' = pav_all_sv_df %>% filter(Len == len) %>% pull(Del),
        'Dup' = pav_all_sv_df %>% filter(Len == len) %>% pull(Dup),
        'Inv' = pav_all_sv_df %>% filter(Len == len) %>% pull(Inv),
        'Ins' = pav_all_sv_df %>% filter(Len == len) %>% pull(INS),
        'Tra' = pav_all_sv_df %>% filter(Len == len) %>% pull(TRA),
        'Unk' = pav_all_sv_df %>% filter(Len == len) %>% pull(UNK)
    )

    par(mar=c(5, 8, 4, 2))
    boxplot(l, las=1, cex=1.3, cex.axis=1.3, frame=FALSE, main=paste("PAV (", len, ")", sep=""), ylim=c(0, 20000))
    mtext("Counts", 2, line=5, cex=1.3)
}

In [ ]:
ha_sample_t2t_df

In [ ]:
# Same for PBSV.
for (file in (ha_sample_t2t_df %>% filter(!is.na(pbsv_structural_variant_stats)) %>% pull(pbsv_structural_variant_stats))) {
    print(file)
}

In [ ]:
# Location of this notebook's data (a gs:// location). Set AOU_LR_DATA before running; see GETTING_STARTED.md.
AOU_LR_DATA <- Sys.getenv("AOU_LR_DATA")
if (AOU_LR_DATA == "") stop("Set the AOU_LR_DATA environment variable before running this notebook")

In [ ]:
system(paste0("gsutil -m cp ", AOU_LR_DATA, "/outputs/T2T/SummarizeStructuralVariants/*/*.pbsv.stats.txt ."), intern=TRUE)

In [ ]:
pbsv_all_sv_df = NULL
for (file in (ha_sample_t2t_df %>% filter(!is.na(ha_sample_t2t_df$pbsv_structural_variant_stats)) %>% pull(pbsv_structural_variant_stats))) {
    pbsv_file = basename(file)
    if (file.exists(pbsv_file)) {
        pbsv_df = read_table(pbsv_file, show_col_types=FALSE)
        pbsv_df = pbsv_df %>% tail(n=4) %>% adorn_totals("row") %>% mutate(sample = pbsv_file)
        
        if (is.null(pbsv_all_sv_df)) {
            pbsv_all_sv_df = pbsv_df
        } else {
            pbsv_all_sv_df = rbind(pbsv_all_sv_df, pbsv_df)
        }
    }
}

In [ ]:
# PBSV counts by type across samples, per length bin.
for (len in pbsv_df$Len) {
    l = list(
        'Del' = pbsv_all_sv_df %>% filter(Len == len) %>% pull(Del),
        'Dup' = pbsv_all_sv_df %>% filter(Len == len) %>% pull(Dup),
        'Inv' = pbsv_all_sv_df %>% filter(Len == len) %>% pull(Inv),
        'Ins' = pbsv_all_sv_df %>% filter(Len == len) %>% pull(INS),
        'Tra' = pbsv_all_sv_df %>% filter(Len == len) %>% pull(TRA),
        'Unk' = pbsv_all_sv_df %>% filter(Len == len) %>% pull(UNK)
    )

    par(mar=c(5, 8, 4, 2))
    boxplot(l, las=1, cex=1.3, cex.axis=1.3, frame=FALSE, main=paste("PBSV (", len, ")", sep=""), ylim=c(0, 20000))
    mtext("Counts", 2, line=5, cex=1.3)
}

In [ ]:
# Same for Sniffles. The per-sample table sniffles_all_sv_df (and sniffles_df) must be built as for PBSV
# above; that cell is not in this copy.
for (file in (ha_sample_t2t_df %>% filter(!is.na(sniffles_structural_variant_stats)) %>% pull(sniffles_structural_variant_stats))) {
    print(file)
}

In [ ]:
system(paste0("gsutil -m cp ", AOU_LR_DATA, "/outputs/T2T/SummarizeStructuralVariants/*/*.sniffles.stats.txt ."), intern=TRUE)

In [ ]:
for (len in sniffles_df$Len) {
    l = list(
        'Del' = sniffles_all_sv_df %>% filter(Len == len) %>% pull(Del),
        'Dup' = sniffles_all_sv_df %>% filter(Len == len) %>% pull(Dup),
        'Inv' = sniffles_all_sv_df %>% filter(Len == len) %>% pull(Inv),
        'Ins' = sniffles_all_sv_df %>% filter(Len == len) %>% pull(INS),
        'Tra' = sniffles_all_sv_df %>% filter(Len == len) %>% pull(TRA),
        'Unk' = sniffles_all_sv_df %>% filter(Len == len) %>% pull(UNK)
    )

    par(mar=c(5, 8, 4, 2))
    boxplot(l, las=1, cex=1.3, cex.axis=1.3, frame=FALSE, main=paste("Sniffles (", len, ")", sep=""), ylim=c(0, 20000))
    mtext("Counts", 2, line=5, cex=1.3)
}

In [ ]:
# Compare the three callers' counts for one sample, per length bin.
rbind(pav_df, pbsv_df, sniffles_df) %>% filter(Len == '50-100bp') %>% select(sample, Del, Dup, Inv, INS, TRA, UNK) %>% as.matrix()


In [ ]:
for (len in pbsv_df$Len) {
    print(len)
    rbind(pav_df, pbsv_df, sniffles_df) %>% filter(Len == len) %>% select(sample, Del, Dup, Inv, INS, TRA, UNK) %>% print()
}


In [ ]:
par(mar=c(5, 7, 4, 2))
for (len in pbsv_df$Len) {
    rbind(pbsv_df, sniffles_df, pav_df) %>% filter(Len == len) %>% select(Del, Dup, Inv, INS, TRA, UNK) %>% as.matrix() %>% barplot(cex.names=1.3, cex.axis=1.3, las=1, beside=TRUE, main=len, col=c("darkred", "darkgreen", "blue"))
    legend("topright", c("PBSV", "Sniffles", "PAV"), fill=c("darkred", "darkgreen", "blue"))
}


In [ ]:
# Total SVs by type for each caller (stacked bars).
par(mar=c(5, 7, 4, 2))

rbind(
    pbsv_df %>% tail(n=1) %>% select(Del, Dup, Inv, INS, TRA, UNK) %>% as.matrix(),
    sniffles_df %>% tail(n=1) %>% select(Del, Dup, Inv, INS, TRA, UNK) %>% as.matrix(),
    pav_df %>% tail(n=1) %>% select(Del, Dup, Inv, INS, TRA, UNK) %>% as.matrix()
) %>% t() %>% barplot(las=1, cex.axis=1.3, names.arg=c("PBSV", "Sniffles", "PAV"), col=c("#FCFFF5", "#D1DBBD", "#91AA9D", "#3E606F", "#193441"))

legend("topright", c("Del", "Dup", "Inv", "Ins", "Tra", "Unk"), fill=c("#FCFFF5", "#D1DBBD", "#91AA9D", "#3E606F", "#193441"))